# Pretrain Commutative Transformer Encoder

Load the shared unlabeled pretraining dataset and save commutative transformer encoder weights for downstream transformer experiments.

In [5]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.ml import CommutativeTransformerClassifier, CommutativeTransformerConfig, LossWeightConfig, OptimizationConfig
from src.tensor_utils import load_unlabeled_tensor_dataset


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
# User inputs

unlabeled_dataset_path = Path(".dataset_cache/unlabeled_active_high_mid_low_t20_z5_y96_x96_chunks")
pretrained_encoder_path = Path("artifacts/pretrained_commutative_transformer/encoder_state.pt")

model_config = CommutativeTransformerConfig(
    spatial_patch_size_st=(1, 32, 32),
    spatial_patch_size_ts=(1, 32, 32),
    temporal_patch_size_ts=2,
    embed_dim=32,
    num_heads=2,
    mlp_ratio=2.0,
    dropout=0.3,
    attention_dropout=0.1,
    st_spatial_depth=1,
    st_temporal_depth=1,
    ts_temporal_depth=1,
    ts_spatial_depth=1,
    embedding_dim=16,
    num_prototypes=8,
)
optimization_config = OptimizationConfig(
    batch_size=8,
    epochs=75,
    learning_rate=1e-4,
    weight_decay=3e-3,
    early_stopping_patience=8,
    early_stopping_min_delta=0.0,
    scheduler_patience=3,
    scheduler_factor=0.7,
    scheduler_min_lr=1e-6,
    validation_split=0.0,
    random_state=0,
    standardize=True,
    device=None,
    verbose=True,
)
loss_weight_config = LossWeightConfig(
    consistency_weight=1.0,
    feature_weight=0.05,
    prototype_temperature=0.1,
)

In [7]:
unlabeled_dataset = load_unlabeled_tensor_dataset(unlabeled_dataset_path)
unlabeled_dataset["tensors"].shape, unlabeled_dataset["metadata"].shape

(torch.Size([100, 20, 5, 96, 96]), (100, 7))

In [8]:
model = CommutativeTransformerClassifier(
    model_config=model_config,
    optimization_config=optimization_config,
    loss_weight_config=loss_weight_config,
)
model.pretrain(unlabeled_dataset["tensors"])
pretrained_encoder_path = model.save_pretrained_encoder(pretrained_encoder_path)
pretrained_encoder_path

cols:
    ep=epoch
    lr=learning_rate
    eta=estimated_time_remaining
    trL=train_loss
    trCC=train_commutative_consistency_loss
    trFA=train_feature_alignment_loss
     ep       lr       eta |      trL     trCC     trFA
001/075 1.00e-04     27:41 |   2.3723   2.3673   0.0997
002/075 1.00e-04     26:07 |   1.7561   1.7529   0.0642
003/075 1.00e-04     25:38 |   1.8515   1.8490   0.0490
004/075 1.00e-04     25:07 |   1.9165   1.9144   0.0414
005/075 1.00e-04     24:24 |   1.9066   1.9051   0.0304
006/075 7.00e-05     24:12 |   1.9502   1.9489   0.0262
007/075 7.00e-05     23:48 |   1.9927   1.9915   0.0231
008/075 7.00e-05     23:28 |   2.0154   2.0144   0.0207
009/075 7.00e-05     23:02 |   2.0064   2.0055   0.0193
010/075 4.90e-05     22:43 |   2.0264   2.0255   0.0178
early_stop epoch=010 best_epoch=002 best_metric=1.7561


PosixPath('artifacts/pretrained_commutative_transformer/encoder_state.pt')

In [9]:
model.pretrain_history_.tail()

,epoch,train_loss,train_commutative_consistency_loss,train_feature_alignment_loss
5,6,1.950218,1.948909,0.026193
6,7,1.992673,1.991519,0.023081
7,8,2.015424,2.014389,0.020707
8,9,2.006431,2.005467,0.019265
9,10,2.026359,2.025471,0.017758
